## 1. Install & Import

In [6]:
!pip install pycaret

Defaulting to user installation because normal site-packages is not writeable
  Using cached pycaret-3.3.2-py3-none-any.whl.metadata (17 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + C:\Program Files\Python313\python.exe C:\Users\BIT\AppData\Local\Temp\pip-install-gk1qxe_7\numpy_08ab735576b64705aa6d2d0fd32d45c6\vendored-meson\meson\meson.py setup C:\Users\BIT\AppData\Local\Temp\pip-install-gk1qxe_7\numpy_08ab735576b64705aa6d2d0fd32d45c6 C:\Users\BIT\AppData\Local\Temp\pip-install-gk1qxe_7\numpy_08ab735576b64705aa6d2d0fd32d45c6\.mesonpy-l1ppea60 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\BIT\AppData\Local\Temp\pip-install-gk1qxe_7\numpy_08ab735576b64705aa6d2d0fd32d45c6\.mesonpy-l1ppea60\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\BIT\AppData\Local\Temp\pip-install-gk1qxe_7\numpy_08ab735576b64705aa6d2d0fd32d45c6
      Build dir: C:\Users\BIT\AppData\Local\Temp\pip-install-gk1qxe_7\numpy_08ab735576b64705aa6d2d0fd32d

In [8]:
import pandas as pd
import numpy as np
from pycaret.classification import *

ModuleNotFoundError: No module named 'pycaret'

## 2. Load Data

In [ ]:
# ⚠️ Update paths if running in Colab (e.g., mount Drive first)
train_df = pd.read_csv(r"../data/FraudTrain.csv")
test_df  = pd.read_csv(r"../data/FraudTest.csv")

print('Train shape:', train_df.shape)
print('Test shape: ', test_df.shape)

## 3. Feature Engineering
Same features as the original notebook: hour, age, distance.

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Drop irrelevant columns
    cols_to_drop = ['Unnamed: 0', 'cc_num', 'first', 'last', 'street', 'trans_num']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # Extract hour from transaction time
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df = df.drop(columns=['trans_date_trans_time'])

    # Age from date of birth
    df['dob'] = pd.to_datetime(df['dob'])
    df['age'] = 2026 - df['dob'].dt.year
    df = df.drop(columns=['dob'])

    # Distance between customer and merchant
    df['distance'] = np.sqrt(
        (df['lat'] - df['merch_lat'])**2 +
        (df['long'] - df['merch_long'])**2
    )

    return df

train_df = engineer_features(train_df)
test_df  = engineer_features(test_df)

print('Features after engineering:', list(train_df.columns))
train_df.head()

## 4. PyCaret Setup
PyCaret automatically handles:
-  Label encoding of categorical columns
-  StandardScaler normalization
-  SMOTE for class imbalance
-  Train/validation splitting

In [9]:
clf = setup(
    data            = train_df,
    target          = 'is_fraud',
    test_data       = test_df,          # use your original test set

    # Preprocessing
    normalize       = True,             # replaces StandardScaler
    fix_imbalance   = True,             # replaces SMOTE

    # Categorical columns (replaces LabelEncoder)
    categorical_features = ['merchant', 'category', 'gender', 'city', 'state', 'job'],

    # Evaluation
    fold            = 5,                # 5-fold cross-validation
    session_id      = 42,
    verbose         = True
)

NameError: name 'setup' is not defined

## 5. Compare All Models


In [ ]:
best_models = compare_models(
    include   = ['rf', 'xgboost', 'lr', 'dt', 'lightgbm'],  # include your original models + extras
    sort      = 'AUC',
    n_select  = 3       # returns top 3 models
)

## 6. Train Specific Models (matching your original notebook)
Recreate RandomForest and XGBoost exactly as before, but through PyCaret.

In [ ]:
# Random Forest  (matches your model1)
model1 = create_model('rf', fold=5)

# XGBoost  (matches your model2)
model2 = create_model('xgboost', fold=5)

## 7. Evaluate Models
PyCaret plots confusion matrix, ROC curve, feature importance — all in one call.

In [ ]:
# --- Random Forest evaluation ---
print('\n===== Random Forest =====')
evaluate_model(model1)

In [ ]:
# --- XGBoost evaluation ---
print('\n===== XGBoost =====')
evaluate_model(model2)

## 8. Detailed Plots

In [ ]:
# Confusion Matrix
plot_model(model1, plot='confusion_matrix', plot_kwargs={'percent': False})
plot_model(model2, plot='confusion_matrix', plot_kwargs={'percent': False})

In [ ]:
# ROC Curve
plot_model(model1, plot='auc')
plot_model(model2, plot='auc')

In [ ]:
# Feature Importance
plot_model(model1, plot='feature')
plot_model(model2, plot='feature')

## 9. Predict on Test Set

In [ ]:
# Finalize trains the model on the full dataset (train + validation)
final_model1 = finalize_model(model1)
final_model2 = finalize_model(model2)

# Predict on the held-out test_df
predictions_rf  = predict_model(final_model1, data=test_df)
predictions_xgb = predict_model(final_model2, data=test_df)

print('RandomForest predictions:')
print(predictions_rf[['prediction_label', 'prediction_score']].head())

print('\nXGBoost predictions:')
print(predictions_xgb[['prediction_label', 'prediction_score']].head())

## 10. Save Models to Google Drive
So you never need to retrain after a Colab disconnect.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/fraud_model_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

save_model(final_model1, f'{SAVE_DIR}/rf_model')
save_model(final_model2, f'{SAVE_DIR}/xgb_model')
print('✅ Both models saved to Google Drive!')

## 11. Load Models After Disconnect
Run this cell instead of retraining after a Colab disconnect.

In [ ]:
# Run this after reconnecting — skips all training!
from pycaret.classification import load_model, predict_model

SAVE_DIR = '/content/drive/MyDrive/fraud_model_checkpoints'

loaded_rf  = load_model(f'{SAVE_DIR}/rf_model')
loaded_xgb = load_model(f'{SAVE_DIR}/xgb_model')

print('✅ Models loaded! Ready to predict.')